# <ins><b> Our Model</b></ins>

In [305]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns
import tensorflow as tf
import tensorflow_probability as tfp


import re
import os
import copy
import warnings
import functools
import contextlib
import time

from scipy import stats
from decimal import Decimal
from datetime import datetime
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import TomekLinks
from bayes_opt import BayesianOptimization
from sklearn.metrics import confusion_matrix, classification_report, fbeta_score
from itertools import product


In [306]:
plt.style.use('dark_background')
pd.set_option("display.precision", 5)

tf.get_logger().setLevel('INFO')

seed = 1
tf.keras.utils.set_random_seed(seed)
tf.random.set_seed(seed)
tf.config.experimental.enable_op_determinism()
np.random.seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

In [307]:
data = pd.read_csv('input_data.csv')
data.head()

,HOUSEID,PERSONID,TRPTRANS,TRPACCMP,DRVRCNT,EDUC,HHFAMINC,HHSIZE,HHSTATE,HHVEHCNT,...,Employ_status,Gender,Norm_tripmile,Norm_triptime,triptime_scaled,triptime_log,IndivID,ALT_16,ALT_23,ALT_45
0,30000007,2,6,0,3,3,7,3,NC,5,...,1,1,0.02870,0.11344,1.2,2.07918,300000072,-1,-1,-1
1,30000012,1,3,0,1,5,10,1,NY,2,...,1,0,0.00147,0.01811,0.2,1.30103,300000121,3,1,3
2,30000039,1,4,0,2,5,10,2,PA,2,...,1,1,0.00394,0.01811,0.2,1.30103,300000391,1,4,4
3,30000041,2,3,0,2,1,11,2,CA,2,...,1,0,0.02663,0.11344,1.2,2.07918,300000412,3,4,3
4,30000062,1,3,1,2,4,10,2,TX,6,...,1,0,0.02062,0.06578,0.7,1.84510,300000621,4,3,4


In [308]:
output_categories = ['DA', 'SR', 'Transit', 'Bicycle', 'Walk']
base_category = 'DA'

asc_variable_names = {category:[] for category in output_categories}
avail_variable_names = {category:f'av_{category}' for category in output_categories}

cols_to_standardize = []
for category in output_categories:
    cols_to_standardize.extend(asc_variable_names[category])

In [309]:
y = np.array(data[output_categories].values.reshape(len(data[output_categories]), len(output_categories)), dtype=np.float32)
X = data.drop(output_categories, axis=1)

In [310]:
generic_variable_names = ['Gender']

asc_variable_indexer = {category:[] for category in output_categories}

avail_variable_indexer = dict(zip(avail_variable_names.keys(), 
                                  X.columns.get_indexer(avail_variable_names.values())))

generic_variable_indexer = dict(zip(generic_variable_names, 
                                  X.columns.get_indexer(generic_variable_names)))

# <ins><b>Model Formulation</b></ins>

In [311]:
# Helper functions

def make_val_and_grad_fn(value_fn):
    @functools.wraps(value_fn)
    def val_and_grad(x):
        return tfp.math.value_and_gradient(value_fn, x)
    return val_and_grad

@contextlib.contextmanager
def timed_execution():
    t0 = time.time()
    yield
    dt = time.time() - t0
    print('Evaluation took: %f seconds' % dt)


def np_value(tensor):
    """Get numpy value out of possibly nested tuple of tensors."""
    if isinstance(tensor, tuple):
        return type(tensor)(*(np_value(t) for t in tensor))
    elif isinstance(tensor, list):
        return [t.numpy()[0] for t in tensor]
    else:
        return tensor.numpy()

def run(optimizer):
    """Run an optimizer and measure it's evaluation time."""
    optimizer()
    with timed_execution():
        result = optimizer()
    return np_value(result)

class Results():
    def __init__(self):
        position = []

        
class HistoryLogger(tf.keras.callbacks.Callback):
    def __init__(self):
        super(HistoryLogger, self).__init__()
        self.history = {}
    
    def on_train_begin(self, logs=None):
        self.history = {}
    
    def on_epoch_begin(self, epoch, logs=None):
        for key, value in logs.items():
            self.history.setdefault(key, []).append(value)
    
    def on_epoch_end(self, epoch, logs=None):
        for key, value in logs.items():
            self.history.setdefault(key, []).append(value)
    
    def on_train_end(self, logs=None):
        if logs.get('plot_loss', False):
            plt.plot(np.arange(len(self.history['loss_train'])), self.history['loss_train'], label='train loss')
        if logs.get('plot_f2_train', False):
            plt.plot(np.arange(len(self.history['f2_train'])), self.history['f2_train'], label='train f2')
        if logs.get('plot_f2_val', False):
            plt.plot(np.arange(len(self.history['f2_val'])), self.history['f2_val'], label='validation f2')
            plt.legend()
        
history = HistoryLogger()

In [423]:
def generate_avail_data(X_tensor):
    avail_vals = []
    for cat in output_categories:
        if cat in avail_variable_names:
            avail_vals.append(X_tensor[:, avail_variable_indexer[cat]])
            avail_vals[-1] = avail_vals[-1].astype(np.float32)
        else:
            avail_vals.append(tf.ones(X_tensor.shape[0], dtype='float32'))
    return tf.stack(avail_vals)



def create_alt_spec_params(alternatives, name, gu_values, val_count):
    alt_spec_vars = []
    for alternative in alternatives:
        alt_spec_var = tf.Variable([gu_values[val_count]], dtype='float32', 
                        name=f'{name}_{alternative}', trainable=True)
        val_count += 1
        alt_spec_vars.append(alt_spec_var)
    return alt_spec_vars, val_count



def initiate_params(alt_spec_vars=True, ind_spec_vars_list=None, alt_spec_constants=True):
    model_params = []
    
    gu_values = tf.abs(tf.keras.initializers.glorot_uniform(seed=1)(shape=(200,))*0)
    val_count = 0
    
    if alt_spec_vars:
        # Generic parameters for alternative specific variables of travelcost, traveltime, travelincentive
        gen_tc = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='travelcost', trainable=True)
        val_count += 1
        gen_tt = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='traveltime', trainable=True)
        val_count += 1
        gen_ti = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='travelincentive', trainable=True)
        val_count += 1
        model_params.extend([gen_tc, gen_tt, gen_ti])
        
    if alt_spec_constants:
        # Alternative specific constants for SR, Transit, Bicycle, Walk
        asc_SR = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='SR_constant', trainable=True)
        val_count += 1
        asc_Transit = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='Transit_constant', trainable=True)
        val_count += 1
        asc_Bicycle = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='Bicycle_constant', trainable=True)
        val_count += 1
        asc_Walk = tf.Variable([gu_values[val_count]], dtype=tf.float32, name='Walk_constant', trainable=True)
        val_count += 1
        model_params.extend([asc_SR, asc_Transit, asc_Bicycle, asc_Walk])
    
    if ind_spec_vars_list is not None:
        alternatives = [cat for cat in output_categories if cat != base_category]
        for var in ind_spec_vars_list:
            isv_asp_params, val_count = create_alt_spec_params(alternatives, var, gu_values, val_count)
            model_params.extend(isv_asp_params)
    
    return model_params


def utilities(x_name, x_val, params, alt_spec_constants=True, ind_spec_vars_list=None):
    x_val = tf.convert_to_tensor(x_val, dtype='float32')
    ntrips = x_val.shape[-1]
    X = dict()
    x_val_trans = tf.transpose(x_val)
    for i, name in enumerate(x_name):
        val = x_val_trans[i]
        X[name] = val
            
    if alt_spec_constants:        
        utility_DA = 0*params[0]*tf.ones_like(X[x_name[0]])
        utility_SR = params[0]*tf.ones_like(X[x_name[0]])
        utility_Transit = params[1]*tf.ones_like(X[x_name[0]])
        utility_Bicycle = params[2]*tf.ones_like(X[x_name[0]])
        utility_Walk = params[3]*tf.ones_like(X[x_name[0]])

    if ind_spec_vars_list is not None:
        nparams = len(params)
        param_count = 0
        var_count = 0
        for i in range(4, nparams):
            isv_asp_param = params[i]
            var_name = ind_spec_vars_list[var_count]
            isv_util = X[var_name] * isv_asp_param
            
            utility_DA = 0*isv_util
            utility_SR += tf.where(tf.equal(param_count, 0), isv_util, 0.0)
            utility_Transit += tf.where(tf.equal(param_count, 1), isv_util, 0.0)
            utility_Bicycle += tf.where(tf.equal(param_count, 2), isv_util, 0.0)
            utility_Walk += tf.where(tf.equal(param_count, 3), isv_util, 0.0)

            param_count += 1
            param_count = param_count % 4
            if param_count == 0:
                var_count += 1
    
    
    
    return tf.transpose(tf.stack((utility_DA, utility_SR, utility_Transit, utility_Bicycle, utility_Walk)))


In [437]:
def get_logits(x_name, x_val, params, alt_spec_constants=True, ind_spec_vars_list=None, layers_list = [], debug=False):
    utils = utilities(x_name, x_val, params, alt_spec_constants=alt_spec_constants, 
                      ind_spec_vars_list=ind_spec_vars_list)
    if debug:
        print("utils:", utils)
        print("-----------------------")
    
    for i, dense_layer in enumerate(layers_list):
        utils = dense_layer(utils)
        if debug:
            print(f"Non-linear layer {i}:", utils)
            print("------------------------------")
    
    return utils


def loss(true_y, logits, avail_vals, null_loglike=False):
    initial_probs = tf.nn.softmax(logits)
    masked_probs = initial_probs*tf.transpose(avail_vals)
    final_probs = tf.divide(masked_probs, 
                            tf.reshape((tf.reduce_sum(masked_probs, axis=-1) + 
                                        tf.keras.backend.epsilon()), 
                                       [len(masked_probs), 1])
                           )
    if null_loglike:
        final_probs = tf.where(final_probs!=0, 1, final_probs)
        final_probs = tf.divide(final_probs,
                                tf.reshape(tf.reduce_sum(final_probs, axis=-1)+tf.keras.backend.epsilon(),
                                          [-1, 1])
                               )
    ce_loss = -tf.reduce_mean(tf.reduce_sum(true_y*tf.math.log(
        final_probs + tf.keras.backend.epsilon()), axis=-1))
    return ce_loss


def apply_regularization(loss, params, l1_lambda=0.001, l2_lambda=0.001):
    # Applying l2 penalty on parameters of alternative specific variables
    l2_penalty = l2_lambda*tf.reduce_sum(tf.math.square(params[:3]))
    
    # Applying l1 penalty on paramters of individual specific variables
    l1_penalty = l1_lambda*tf.reduce_sum(tf.math.abs(params[8:]))
    
    return loss + l1_penalty + l2_penalty


def predict(x_name, x_val, params, avail_vals, alt_spec_constants=True, ind_spec_vars_list=None, layers_list=[]):
    """
    Returns the final probabilites and mode predictions
    """
    utils = get_logits(x_name, x_val, params, alt_spec_constants=alt_spec_constants, 
                       ind_spec_vars_list=ind_spec_vars_list, layers_list=layers_list)
    initial_probs = tf.nn.softmax(utils)
    masked_probs = initial_probs*tf.transpose(avail_vals)
    final_probs = tf.divide(masked_probs, 
                            tf.reshape((tf.reduce_sum(masked_probs, axis=-1) + tf.keras.backend.epsilon()), 
                                       [-1, 1])
                           )
    
    return final_probs, tf.argmax(final_probs, axis=1)

## Training

In [438]:
# Optimizer functions for BFGS method
def mnl_bfgs(l1_lambda, l2_lambda):
    
    @make_val_and_grad_fn
    def loss_val_nd_loss_grad(params):
        curr_utils = utilities(x_names, x_vals, params, ind_spec_vars_list=ind_spec_vars, alt_spec_constants=True)
        curr_loss = loss(ohe_true_y, curr_utils, avail_vals)
        penalty_loss = apply_regularization(curr_loss, params, l1_lambda=l1_lambda, l2_lambda=l2_lambda)
        return penalty_loss
    
    def mnl_with_bfgs():
        return tfp.optimizer.bfgs_minimize(loss_val_nd_loss_grad, 
                                          initial_position=tf.reshape(params, shape=[-1]),
                                          tolerance=1e-8,
                                          max_iterations=500)
    return mnl_with_bfgs()
        

def outer_mnl_bfgs(l2_lambda, l1_lambda):
    l1_lambda = tf.convert_to_tensor(l1_lambda, dtype=tf.float32)
    l2_lambda = tf.convert_to_tensor(l2_lambda, dtype=tf.float32)
    
    result_tensor = mnl_bfgs(l1_lambda, l2_lambda)
    
    results = np_value(result_tensor)    
    return results

In [439]:
ind_spec_vars = ['Gender']

config_dict = {
    'param intialization': 'zero initialization',
    'evaluation metric': 'F2',
    'ind_spec_vars': ind_spec_vars,
    'lambda1': 0,
    'lambda2': 0
}

In [440]:
params = initiate_params(ind_spec_vars_list=ind_spec_vars, alt_spec_vars=False)
avail_vals = generate_avail_data(X.values)
x_vals = X[ind_spec_vars].values
x_names = X[ind_spec_vars].columns
ohe_true_y = y

In [441]:
results = outer_mnl_bfgs(0, 0)

In [442]:
print("BFGS Results")
print("Converged:", results.converged)
print("Iterations:", results.num_iterations)

std_err = np.sqrt(np.diag(pd.DataFrame(results.inverse_hessian_estimate)))/np.sqrt(len(y))
t_ratio = results.position/std_err

pd.DataFrame({'Variable': [param.name[:-2] for param in params], 
              'Coef': results.position,
             'Std.err': std_err,
             't-ratio': t_ratio})

BFGS Results
Converged: True
Iterations: 22


,Variable,Coef,Std.err,t-ratio
0,SR_constant,-1.31799,0.02203,-59.81881
1,Transit_constant,-2.97071,0.06459,-45.99421
2,Bicycle_constant,-3.79504,0.12677,-29.93631
3,Walk_constant,-3.17402,0.07267,-43.67739
4,Gender_SR,-0.08822,0.03200,-2.75716
5,Gender_Transit,0.07636,0.08822,0.86549
6,Gender_Bicycle,0.58647,0.15109,3.88149
7,Gender_Walk,-0.20317,0.10281,-1.97622


In [443]:
results.inverse_hessian_estimate

array([[  19.504038 ,    6.2268095,    5.7856126,    6.565996 ,
         -19.479937 ,   -6.618767 ,   -5.370468 ,   -7.0442977],
       [   6.2268095,  167.60718  ,   24.63774  ,   32.646828 ,
          -6.259985 , -167.5435   ,  -25.037315 ,  -32.33532  ],
       [   5.7856126,   24.63774  ,  645.6731   ,   31.36983  ,
          -6.135289 ,  -24.43666  , -646.8143   ,  -32.11664  ],
       [   6.565996 ,   32.646828 ,   31.36983  ,  212.16989  ,
          -6.7068815,  -33.184906 ,  -33.110867 , -211.42104  ],
       [ -19.479937 ,   -6.259985 ,   -6.135289 ,   -6.7068815,
          41.13344  ,   14.008112 ,   13.272326 ,   14.975272 ],
       [  -6.618767 , -167.5435   ,  -24.43666  ,  -33.184906 ,
          14.008112 ,  312.71753  ,   48.769356 ,   63.07309  ],
       [  -5.370468 ,  -25.037315 , -646.8143   ,  -33.110867 ,
          13.272326 ,   48.769356 ,  917.21704  ,   60.408447 ],
       [  -7.0442977,  -32.33532  ,  -32.11664  , -211.42104  ,
          14.975272 ,   63.07309 

# <ins><b> Kim's Model </b></ins>

In [444]:
import numpy
import tensorflow as tf
import tensorflow_probability as tfp
import pandas as pd
from matplotlib import pyplot
import numpy as np

In [445]:
data = pd.read_csv('Input_data.csv')

In [446]:
# Exogeneous variable
exogen_list = {'gender': data['Gender']}

# Availability variable
avail_list = {'av_DA': data['av_DA'], 
              'av_SR': data['av_SR'],
              'av_Transit': data['av_Transit'],
              'av_Bicycle': data['av_Bicycle'],
              'av_Walk': data['av_Walk']}

avail_val = np.transpose(list(avail_list.values()))

# Endogenous variable
choice_list = ['DA', 'SR', 'Transit', 'Bicycle', 'Walk']

# Define Parameters
params = { #'asc_DA':      tf.constant(tf.zeros([1,]), dtype=tf.float32, name='asc_DA'),
           'asc_SR':      tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='asc_SR'),
           'asc_Transit': tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='asc_Transit'),
           'asc_Bicycle': tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='asc_Bicycle'),
           'asc_Walk':    tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='asc_Walk'),
           #'gender_DA':   tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='gender_DA'),
           'gender_SR':   tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='gender_SR'),
           'gender_transit': tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='gender_transit'),
           'gender_bike': tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='gender_bike'),
           'gender_walk': tf.Variable(tf.zeros([1,]), dtype=tf.float32, name='gender_walk') 
         }

param_name = params.keys()
param_val =  list(params.values())
print([param.numpy()[0] for param in param_val])

x_name = exogen_list.keys()
x_val =  list(exogen_list.values())

# One-hot encode
y_train = np.array(data[choice_list].values.reshape(len(data[choice_list]), len(choice_list)), dtype=np.float32)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [448]:
# Define the utility functions
def utility_func(x_name, x_val, param_name, param_val):
    
    params  = {k:v for k,v in zip(param_name, param_val)}
    x_train = {i:j for i,j in zip(x_name, x_val)}
    
    v1 = tf.zeros([len(x_val[0]),1]) # Baseline
#     v1 = params['gender_DA']*x_train['gender']
    v2 = params['asc_SR'] + params['gender_SR']*x_train['gender']
    v3 = params['asc_Transit'] + params['gender_transit']*x_train['gender']
    v4 = params['asc_Bicycle'] + params['gender_bike']*x_train['gender']
    v5 = params['asc_Walk'] + params['gender_walk']*x_train['gender']
    
#     v1 = tf.reshape(v1, shape=[len(v1), 1])
    v2 = tf.reshape(v2, shape=[len(v2), 1])
    v3 = tf.reshape(v3, shape=[len(v3), 1])
    v4 = tf.reshape(v4, shape=[len(v4), 1])
    v5 = tf.reshape(v5, shape=[len(v5), 1])
    
    return v1, v2, v3, v4, v5

In [449]:
def model_fun(x_name, x_val, param_name, param_val, avail_val):
    
    # Calculate exponential terms with the availability
    utility = utility_func(x_name, x_val, param_name, param_val)
    exp_inv = tf.reshape(tf.transpose(tf.exp(utility)), shape=(len(avail_val), np.size(avail_val,1)))
    ex_v_av = avail_val*exp_inv
    exp_summ = tf.reshape(tf.reduce_sum(ex_v_av, axis=1), shape=(len(avail_val), 1))

    # Calculate probability
    P = ex_v_av/exp_summ
    
    return P

In [450]:
# multi-class cost_entropy
def cost_fun(y_train, yhat):
    
    '''The arbitrary value is added to resolve log(zero)'''
    loss = -tf.reduce_mean(tf.reduce_sum(y_train*tf.math.log(yhat+1e-8), axis=1), axis=0)
    return loss


In [451]:
# obtain the shapes of all trainable parameters in the model
def LL_gradient(x_name, x_val, param_name, param_val, y_train):
    
    shapes = tf.shape_n(param_val)
    n_tensors = len(shapes)

    count = 0
    idx =  [] 
    part = []

    for i, shape in enumerate(shapes):
        n = numpy.product(shape)
        idx.append(tf.reshape(tf.range(count, count+n, dtype=tf.int32), shape))
        part.extend([i]*n)
        count += n
  
    def assign_new_model_parameters(params_1d):
        updated_params = tf.dynamic_partition(params_1d, part, n_tensors)

        for i, (shape, param) in enumerate(zip(shapes, updated_params)):
            param_val[i].assign(tf.reshape(param, shape))

    def est_grad(params_1d):

        # Derive the Tensorflow gradient
        with tf.GradientTape() as tape: 

            # Call the function to update and convert the shape of parameters
            assign_new_model_parameters(params_1d)

            # Estimated Choice Probability 
            yhat = model_fun(x_name, x_val, param_name, param_val, avail_val)

            # Call the cost function
            loss_value = cost_fun(y_train, yhat)

        # Calculate the gradient for each parameter
        estimated_grad = tape.gradient(loss_value, param_val)

        grads_1dim = tf.dynamic_stitch(idx, estimated_grad)
        return loss_value, grads_1dim

    est_grad.idx = idx

    return est_grad

In [452]:
# Define the positions of initial parameters
init_params = tf.dynamic_stitch(LL_gradient(x_name, x_val, param_name, param_val, y_train).idx, param_val)
print(init_params)

# Implement the BFGS optimizer
Trained_Results = tfp.optimizer.bfgs_minimize(
                                      value_and_gradients_function=LL_gradient(x_name, x_val, param_name, param_val, y_train), 
                                      initial_position=init_params,
                                      tolerance=1e-08,
                                      max_iterations=500)

tf.Tensor([0. 0. 0. 0. 0. 0. 0. 0.], shape=(8,), dtype=float32)


In [453]:
est_title = pd.DataFrame(params.keys(), columns=['Variable'])
# Estimated Parameters
est_para = pd.DataFrame(Trained_Results.position.numpy(), columns=['Coef.'])
# Standard Errors
Std_err = pd.DataFrame(np.sqrt(np.diag(pd.DataFrame(Trained_Results.inverse_hessian_estimate.numpy())))/np.sqrt(len(y_train)), columns=['Std.err'])
# t-ratio
t_ratio = pd.DataFrame(est_para.values/Std_err.values, columns=['t-ratio'])
# Estimation results table
Est_result = pd.concat([est_title, est_para, Std_err, t_ratio], axis=1).set_index('Variable')
print(Est_result)

# Loglikelihood Function
LL_initi = tf.reduce_sum(y_train*tf.math.log(model_fun(x_name, x_val, param_name, init_params, avail_val)+1e-8))
LL_final = tf.reduce_sum(y_train*tf.math.log(model_fun(x_name, x_val, param_name, param_val, avail_val)+1e-8))
print("LL(initial):", LL_initi.numpy())
print("LL(final):  ", LL_final.numpy())

# Akaike information criterion (AIC)
Estimated_parameters = len(param_name)
AIC = -2*LL_final+ 2*Estimated_parameters
print("AIC:        ", AIC.numpy())
# Bayesian information criterion (BIC)
BIC = -2*LL_final+ Estimated_parameters*np.log(len(x_val))
print("BIC:        ", BIC.numpy())

                  Coef.  Std.err   t-ratio
Variable                                  
asc_SR         -1.31799  0.02199 -59.93309
asc_Transit    -2.97072  0.06457 -46.00895
asc_Bicycle    -3.79504  0.12680 -29.92924
asc_Walk       -3.17402  0.07267 -43.67849
gender_SR      -0.08822  0.03202  -2.75523
gender_transit  0.07636  0.08825   0.86527
gender_bike     0.58647  0.15105   3.88273
gender_walk    -0.20317  0.10280  -1.97626
LL(initial): -27031.938
LL(final):   -16176.522
AIC:         32369.045
BIC:         32353.045


In [454]:
Trained_Results.num_iterations.numpy()

22

In [455]:
Trained_Results.inverse_hessian_estimate

<tf.Tensor: shape=(8, 8), dtype=float32, numpy=
array([[  19.42973  ,    6.531304 ,    6.164247 ,    6.5985312,
         -19.444492 ,   -6.8594217,   -5.6930375,   -7.079053 ],
       [   6.531304 ,  167.50034  ,   24.625046 ,   32.46187  ,
          -6.6855354, -167.60146  ,  -24.57662  ,  -32.232357 ],
       [   6.164247 ,   24.625046 ,  645.9794   ,   31.143314 ,
          -6.727722 ,  -24.60825  , -646.5029   ,  -32.002277 ],
       [   6.5985312,   32.46187  ,   31.143314 ,  212.15964  ,
          -6.7106566,  -33.02789  ,  -32.935295 , -211.40591  ],
       [ -19.444492 ,   -6.6855354,   -6.727722 ,   -6.7106566,
          41.190605 ,   14.376881 ,   13.66835  ,   15.003505 ],
       [  -6.8594217, -167.60146  ,  -24.60825  ,  -33.02789  ,
          14.376881 ,  312.89026  ,   48.497047 ,   62.994427 ],
       [  -5.6930375,  -24.57662  , -646.5029   ,  -32.935295 ,
          13.66835  ,   48.497047 ,  916.63513  ,   60.300583 ],
       [  -7.079053 ,  -32.232357 ,  -32.002277 ,